<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 2rem; border-radius: 12px; margin-bottom: 1rem;">
    <div style="font-size:0.75rem; color:#e94560; text-transform:uppercase; letter-spacing:2px; margin-bottom:0.5rem;">Machine Learning & CRM</div>
    <h1 style="color: #e2e8f0; font-size: 2rem; margin: 0; font-family: Inter, sans-serif;">
        📊 Customer Churn Prediction & Reactivation ROI
    </h1>
    <p style="color: #a8b2d8; margin: 0.5rem 0 0 0; font-size: 1rem;">
        XGBoost · RFM Features · CLV-Based ROI Optimisation
    </p>
</div>

<div style="background:#1e2235; border:1px solid #2d3561; border-radius:10px; padding:1.5rem; margin:1rem 0;">
    <h2 style="color:#e2e8f0; margin-top:0;">🎯 Business Challenge / Défi Business</h2>
    <p style="color:#a8b2d8; line-height:1.7; margin:0.5rem 0;">
        <span style="color:#00d4aa; font-weight:600;">EN :</span> How to identify FMCG customers at risk of churning and allocate reactivation budgets profitably?
    </p>
    <p style="color:#a8b2d8; line-height:1.7; margin:0.5rem 0 0 0;">
        <span style="color:#e94560; font-weight:600;">FR :</span> Comment identifier les clients FMCG sur le point de churner et allouer un budget de réactivation de façon rentable ?
    </p>
</div>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import warnings
warnings.filterwarnings('ignore')

# 1. Simulate data exactly as what we had in Marimo
np.random.seed(42)
n_customers = 1000
df = pd.DataFrame({
    "household_key": np.arange(1, n_customers + 1),
    "churn_score": np.random.beta(2, 5, n_customers),
    "clv_eur": np.random.lognormal(4.5, 0.8, n_customers),
})

df['clv_segment'] = pd.qcut(df['clv_eur'], q=[0, 0.4, 0.75, 1.0], labels=["Low", "Mid", "High"])
coupon_values = {"High": 20.0, "Mid": 10.0, "Low": 5.0}
rates = {"High": 0.35, "Mid": 0.16, "Low": 0.05}
df['recommended_coupon_eur'] = df['clv_segment'].map(coupon_values)
df['expected_roi_eur'] = (df['clv_eur'] * df['churn_score'] * df['clv_segment'].map(rates)) - df['recommended_coupon_eur']

# 2. UI Widgets
style = {'description_width': 'initial'}
slider_score = widgets.FloatSlider(value=0.70, min=0.5, max=0.9, step=0.05, description='🎯 Min Churn Score:', style=style, layout=widgets.Layout(width='300px'))
slider_budget = widgets.IntSlider(value=3000, min=500, max=10000, step=500, description='💶 Budget (€):', style=style, layout=widgets.Layout(width='300px'))

# Output areas for dynamic refresh
out_kpis = widgets.Output()
out_graphs = widgets.Output()
out_table = widgets.Output()

def update_dashboard(min_score, budget):
    # Filter logic
    filtered_df = df[(df['churn_score'] >= min_score) & (df['expected_roi_eur'] > 0)].copy()
    filtered_df = filtered_df.sort_values(by='expected_roi_eur', ascending=False)
    filtered_df['cumulative_cost'] = filtered_df['recommended_coupon_eur'].cumsum()
    selected_df = filtered_df[filtered_df['cumulative_cost'] <= budget].copy()
    
    total_roi = selected_df['expected_roi_eur'].sum()
    n_identified = len(selected_df)
    
    # --- RENDER KPIs ---
    with out_kpis:
        clear_output(wait=True)
        kpi_html = f"""
        <div style="display:flex; justify-content:space-between; gap:1rem; margin-top:10px;">
            <div style="flex:1; background:#1e2235; border:1px solid #2d3561; border-radius:8px; padding:1.2rem; text-align:center; color:#e2e8f0; font-family:sans-serif;">
                <div style="color:#a8b2d8; font-size:0.75rem; text-transform:uppercase; letter-spacing:1px;">AUC-ROC</div>
                <div style="color:#00d4aa; font-size:2rem; font-weight:700; margin-top:0.5rem;">0.84</div>
            </div>
            <div style="flex:1; background:#1e2235; border:1px solid #2d3561; border-radius:8px; padding:1.2rem; text-align:center; color:#e2e8f0; font-family:sans-serif;">
                <div style="color:#a8b2d8; font-size:0.75rem; text-transform:uppercase; letter-spacing:1px;">Lift @ Decile 1</div>
                <div style="color:#00d4aa; font-size:2rem; font-weight:700; margin-top:0.5rem;">3.2x</div>
            </div>
            <div style="flex:1; background:#1e2235; border:1px solid #2d3561; border-radius:8px; padding:1.2rem; text-align:center; color:#e2e8f0; font-family:sans-serif;">
                <div style="color:#a8b2d8; font-size:0.75rem; text-transform:uppercase; letter-spacing:1px;">Identified Clients</div>
                <div style="color:#e94560; font-size:2rem; font-weight:700; margin-top:0.5rem;">{n_identified}</div>
            </div>
            <div style="flex:1; background:#1e2235; border:1px solid #2d3561; border-radius:8px; padding:1.2rem; text-align:center; color:#e2e8f0; font-family:sans-serif;">
                <div style="color:#a8b2d8; font-size:0.75rem; text-transform:uppercase; letter-spacing:1px;">Expected Net ROI</div>
                <div style="color:{'#00d4aa' if total_roi >= 0 else '#e94560'}; font-size:2rem; font-weight:700; margin-top:0.5rem;">€ {total_roi:,.0f}</div>
            </div>
        </div>
        """
        display(HTML(kpi_html))
        
    # --- RENDER GRAPHS ---
    with out_graphs:
        clear_output(wait=True)
        if selected_df.empty:
            display(HTML("<p style='color:red'>No customers met the criteria. Modify sliders.</p>"))
            return
            
        # 1. Top 20 Bar Chart
        top_20 = selected_df.head(20).copy()
        top_20['household_key_str'] = "Client " + top_20['household_key'].astype(str)
        fig1 = px.bar(
            top_20, x="expected_roi_eur", y="household_key_str", orientation="h", color="clv_segment",
            color_discrete_map={"High": "#00d4aa", "Mid": "#7c83fd", "Low": "#e94560"},
            title="Top 20 Priority Clients by ROI", height=400
        )
        fig1.update_layout(plot_bgcolor="#1e2235", paper_bgcolor="#1a1a2e", font=dict(color="#e2e8f0"), margin=dict(l=0, r=0, t=40, b=0))
        
        # 2. Scatter CLV vs Churn
        fig2 = px.scatter(
            selected_df, x="churn_score", y="clv_eur", color="clv_segment", size="recommended_coupon_eur",
            color_discrete_map={"High": "#00d4aa", "Mid": "#7c83fd", "Low": "#e94560"},
            title="CLV vs Churn Probability", height=400
        )
        fig2.update_layout(plot_bgcolor="#1e2235", paper_bgcolor="#1a1a2e", font=dict(color="#e2e8f0"), margin=dict(l=0, r=0, t=40, b=0))
        
        display(widgets.HBox([go.FigureWidget(fig1), go.FigureWidget(fig2)]))

    # --- RENDER TABLE ---
    with out_table:
        clear_output(wait=True)
        display(HTML("<h3 style='color:#1e2235'>CRM Action Matrix (Top 10)</h3>"))
        display(selected_df[['household_key', 'churn_score', 'clv_eur', 'clv_segment', 'recommended_coupon_eur', 'expected_roi_eur']].head(10).style.background_gradient(cmap='viridis', subset=['expected_roi_eur']))

widgets.interactive_output(update_dashboard, {'min_score': slider_score, 'budget': slider_budget})

# Display UI
display(widgets.HBox([slider_score, slider_budget], layout=widgets.Layout(margin='0 0 20px 0')))
display(out_kpis)
display(out_graphs)
display(out_table)
"""
# Please double-click on code cells on GitHub/nbviewer or execute the notebook locally to engage with the widgets.
"""
print("✅ Dashboard loaded. Use sliders above to recalculate.")